# Phase 2 — Fixed-Graph Baseline (Performance Ceiling)

Set `G = gold_standard` (true regulatory graph) and fit dynamics parameters
(W, γ, σ) with PyMC. This is the **performance ceiling**: if the model cannot
fit trajectories well even when given the true graph, the data or ODE
formulation is the problem, not the inference.

**Phase 2a** — MAP estimate (seconds, sanity check)  
**Phase 2b** — NUTS posterior (5–30 min, calibrated credible intervals)

Results feed directly into Paper 2 Section 4 (Fixed-Graph Ceiling).

In [ ]:
import pathlib, sys, warnings
warnings.filterwarnings('ignore')

ROOT = pathlib.Path('../../../../')
sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import arviz as az
import pymc as pm

from grn_world_model.data_loader import load_dream4_network, DREAM4Network
from grn_world_model.ode_model import ode_solve, apply_knockout
from grn_world_model.pymc_model import (
    build_per_particle_model, map_estimate, _build_W,
)
from grn_world_model.jax_model import run_nuts, mcmc_to_arviz

DATA_DIR    = ROOT / 'data'
RESULTS_DIR = ROOT / 'results' / 'grn_world_model' / 'phase2_fixed_graph'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

NETWORK_ID = 1   # change to run other networks (1–5)
SIZE       = 10  # Size10 only for Phase 2 (NUTS tractable)

net = load_dream4_network(DATA_DIR, size=SIZE, network_id=NETWORK_ID)
print(f'Loaded net{NETWORK_ID} (Size{SIZE}): {net.n_genes} genes, '
      f'{int(net.gold_standard.sum())} true edges, '
      f'{len(net.timeseries)} timeseries, {len(net.knockout_pairs)} knockouts')

## 1. Data Exploration

In [ ]:
fig, axes = plt.subplots(1, len(net.timeseries), figsize=(16, 3), sharey=True)
colors = plt.cm.tab10.colors

for i, (ax, ts) in enumerate(zip(axes, net.timeseries)):
    for g in range(net.n_genes):
        ax.plot(ts.timepoints, ts.expression[:, g], color=colors[g],
                lw=1.2, alpha=0.8, label=f'G{g+1}')
    ax.axvline(500, color='k', lw=0.8, ls='--', alpha=0.5)
    ax.set_title(f'Series {i+1}', fontsize=10)
    ax.set_xlabel('Time')

axes[0].set_ylabel('Expression')
axes[-1].legend(fontsize=7, loc='upper right', ncol=2)
fig.suptitle(f'DREAM4 net{NETWORK_ID} — Timeseries (dashed = perturbation end)',
             fontsize=11)
plt.tight_layout()
plt.savefig(RESULTS_DIR / f'net{NETWORK_ID}_timeseries.png', dpi=150)
plt.show()

In [ ]:
# Gold standard adjacency heatmap
fig, ax = plt.subplots(figsize=(4, 4))
im = ax.imshow(net.gold_standard, cmap='Blues', vmin=0, vmax=1)
ax.set_xlabel('Target gene'); ax.set_ylabel('Source gene')
ax.set_title(f'Gold standard G (net{NETWORK_ID}): {int(net.gold_standard.sum())} edges')
ax.set_xticks(range(net.n_genes)); ax.set_yticks(range(net.n_genes))
ax.set_xticklabels([f'G{i+1}' for i in range(net.n_genes)], rotation=90, fontsize=8)
ax.set_yticklabels([f'G{i+1}' for i in range(net.n_genes)], fontsize=8)
plt.tight_layout()
plt.savefig(RESULTS_DIR / f'net{NETWORK_ID}_gold_standard.png', dpi=150)
plt.show()

## 2. Phase 2a — MAP Estimate

In [ ]:
model_map = build_per_particle_model(net.gold_standard, net, use_knockouts=True)
print('Free RVs:', [v.name for v in model_map.free_RVs])
map_pt = map_estimate(model_map)

print(f'\nMAP logp = {model_map.compile_logp()(map_pt):.2f}')
print(f'gamma:     {map_pt["gamma"].round(3)}')
print(f'sigma_obs: {map_pt["sigma_obs"].round(3)}')
print(f'w_flat:    {map_pt["w_flat"].round(3)}')

In [ ]:
# Evaluate MAP trajectory predictions vs observed
from grn_world_model.pymc_model import _build_W
import numpy as np

G = net.gold_standard
active_edges = np.argwhere(G)
W_map = _build_W(map_pt['w_flat'], active_edges, net.n_genes)
gamma_map = map_pt['gamma']

mse_list = []
fig, axes = plt.subplots(1, len(net.timeseries), figsize=(16, 3), sharey=True)
for i, (ax, ts) in enumerate(zip(axes, net.timeseries)):
    traj = ode_solve(W_map, G, gamma_map, net.wildtype,
                     ts.timepoints, ts.expression[0], ts.perturbation)
    mse = float(np.mean((traj - ts.expression)**2))
    mse_list.append(mse)
    for g in range(net.n_genes):
        ax.plot(ts.timepoints, ts.expression[:, g], 'o', ms=3,
                color=colors[g], alpha=0.6)
        ax.plot(ts.timepoints, traj[:, g], '-', lw=1.2, color=colors[g])
    ax.set_title(f'Series {i+1}\nMSE={mse:.4f}', fontsize=9)
    ax.set_xlabel('Time')

axes[0].set_ylabel('Expression')
fig.suptitle(f'MAP trajectory predictions (dots=obs, lines=pred) — net{NETWORK_ID}', fontsize=11)
plt.tight_layout()
plt.savefig(RESULTS_DIR / f'net{NETWORK_ID}_map_trajectories.png', dpi=150)
plt.show()
print(f'Mean MSE across series: {np.mean(mse_list):.4f}')

## 3. Phase 2b — NUTS Posterior

Uses **JAX + diffrax + numpyro**: autodiff through the ODE (no sensitivity equations),
JIT-compiled likelihood, ~3s/draw on CPU vs ~220s/draw with `pm.ode.DifferentialEquation`.

Runtime: ~90–120 min for 2 chains × 500 draws on Size10.

In [ ]:
import time, os
os.environ.setdefault('JAX_PLATFORMS', 'cpu')

t_start = time.time()
mcmc = run_nuts(
    net.gold_standard, net,
    num_samples=500,
    num_warmup=500,
    num_chains=2,
    use_knockouts=True,
    seed=42,
)
elapsed = time.time() - t_start
print(f'Sampling complete in {elapsed/60:.1f} min.')

# Convert to ArviZ InferenceData
idata = mcmc_to_arviz(mcmc)
print(idata)

In [ ]:
# Convergence diagnostics
rhat    = az.rhat(idata)
ess     = az.ess(idata)
rhat_max = float(rhat.to_array().max())
ess_min  = float(ess.to_array().min())

print(f'R-hat max: {rhat_max:.4f}  (target < 1.01)')
print(f'ESS min:   {ess_min:.0f}    (target > 100)')
print()
print(az.summary(idata, var_names=['w_flat', 'gamma', 'sigma_obs'],
                 round_to=3).to_string())

In [ ]:
# Trace plots for key parameters
az.plot_trace(idata, var_names=['gamma', 'sigma_obs'], compact=True)
plt.suptitle(f'NUTS trace — net{NETWORK_ID} fixed graph (JAX/diffrax)', fontsize=11)
plt.tight_layout()
plt.savefig(RESULTS_DIR / f'net{NETWORK_ID}_nuts_trace.png', dpi=150)
plt.show()

In [ ]:
# Posterior over edge weights (w_flat)
import jax.numpy as jnp

samples = mcmc.get_samples()
w_samples = np.array(samples['w_flat'])   # (n_draws*n_chains, n_active)
active_edges = np.argwhere(net.gold_standard)

fig, ax = plt.subplots(figsize=(10, 3))
ax.violinplot([w_samples[:, i] for i in range(len(active_edges))],
              positions=range(len(active_edges)), showmedians=True)
ax.axhline(0, color='k', lw=0.8, ls='--')
ax.set_xlabel('Edge index')
ax.set_ylabel('Weight posterior')
ax.set_title(f'Posterior over edge weights (net{NETWORK_ID}, {len(active_edges)} active edges)')
plt.tight_layout()
plt.savefig(RESULTS_DIR / f'net{NETWORK_ID}_nuts_weights.png', dpi=150)
plt.show()

In [ ]:
# Posterior predictive: manually run ODE for each posterior sample
# Sample 200 draws uniformly from the combined chains for speed
n_plot = 200
n_total = w_samples.shape[0]
idx = np.random.choice(n_total, size=min(n_plot, n_total), replace=False)

gamma_samples = np.array(samples['gamma'])   # (n_total, N)

fig, axes = plt.subplots(1, len(net.timeseries), figsize=(16, 3), sharey=True)
colors = plt.cm.tab10.colors

for i, (ax, ts) in enumerate(zip(axes, net.timeseries)):
    ppc_traj = []
    for j in idx:
        W_j = _build_W(w_samples[j], active_edges, net.n_genes)
        try:
            traj = ode_solve(W_j, net.gold_standard, gamma_samples[j],
                             net.wildtype, ts.timepoints, ts.expression[0], ts.perturbation)
            ppc_traj.append(traj[1:])   # skip t=0 (initial condition)
        except RuntimeError:
            pass

    ppc_traj = np.array(ppc_traj)   # (n_valid, T-1, N)
    for g in range(net.n_genes):
        lo, hi = np.percentile(ppc_traj[:, :, g], [5, 95], axis=0)
        med     = np.median(ppc_traj[:, :, g], axis=0)
        ax.fill_between(ts.timepoints[1:], lo, hi, alpha=0.15, color=colors[g])
        ax.plot(ts.timepoints[1:], med, lw=1, color=colors[g])
        ax.plot(ts.timepoints[1:], ts.expression[1:, g], 'o', ms=2,
                color=colors[g], alpha=0.7)
    ax.set_title(f'Series {i+1}', fontsize=9)
    ax.set_xlabel('Time')

axes[0].set_ylabel('Expression')
fig.suptitle(f'90% posterior predictive intervals — net{NETWORK_ID} (JAX NUTS)', fontsize=11)
plt.tight_layout()
plt.savefig(RESULTS_DIR / f'net{NETWORK_ID}_nuts_ppc.png', dpi=150)
plt.show()

In [ ]:
# Save ArviZ trace
trace_path = RESULTS_DIR / f'net{NETWORK_ID}_nuts_trace.nc'
idata.to_netcdf(str(trace_path))
print(f'Trace saved to {trace_path}')

# Summary record
import json
summary = {
    'network': f'net{NETWORK_ID}',
    'size': SIZE,
    'n_genes': net.n_genes,
    'n_true_edges': int(net.gold_standard.sum()),
    'n_active_edges': len(active_edges),
    'sampler': 'JAX/diffrax/numpyro NUTS',
    'num_samples': 500,
    'num_warmup': 500,
    'num_chains': 2,
    'rhat_max': round(rhat_max, 4),
    'ess_min': round(ess_min, 1),
    'map_mean_mse': round(float(np.mean(mse_list)), 4),
}
with open(RESULTS_DIR / f'net{NETWORK_ID}_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))